# Qwen3 ASR 0.6B - Colab Load Check

This notebook loads `Qwen/Qwen3-ASR-0.6B`, provides a simple microphone/upload workflow, and benchmarks latency and throughput on a Colab T4 GPU.

Use `Runtime -> Change runtime type -> T4 GPU` before running the model cells.

In [ ]:
# Install runtime dependencies. Restart the runtime if Colab asks for it.
# The dataset card recommends datasets==2.14.6 because this dataset still uses an HF dataset script.
!pip install -U -q qwen-asr ipywidgets soundfile librosa pandas datasets==2.14.6 "huggingface_hub>=0.24" pyarrow==12.0.1

In [ ]:
import json
import time

from qwen_asr import Qwen3ASRModel
import pandas as pd
import torch

from src import utils, audio, asr, get_data

from ipywidgets import Button, FloatSlider, Dropdown, FileUpload, HBox, Output, VBox
from IPython.display import Audio, clear_output, display

In [ ]:
MODEL_ID = utils.MODEL_ID_ASR
WORK_DIR = utils.WORK_DIR_ASR
WORK_DIR.mkdir(parents=True, exist_ok=True)


MODEL_DTYPE = utils.choose_dtype()
DEVICE_MAP = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print({'model_id': MODEL_ID, 'device_map': DEVICE_MAP, 'dtype': str(MODEL_DTYPE)})

In [ ]:
print(utils.gpu_snapshot())
!nvidia-smi

In [ ]:
utils.reset_gpu_peak()
load_started = time.perf_counter()
asr_model = Qwen3ASRModel.from_pretrained(
    MODEL_ID,
    dtype=MODEL_DTYPE,
    device_map=DEVICE_MAP,
    max_inference_batch_size=16
)
load_seconds = time.perf_counter() - load_started
print({'load_seconds': round(load_seconds, 3), 'gpu': utils.gpu_snapshot()})

In [ ]:
record_seconds = FloatSlider(value=5, min=1, max=30, step=1, description='Seconds')
language = Dropdown(options=[('Auto', None), ('English', 'English'), ('Spanish', 'Spanish'), ('French', 'French'), ('German', 'German')], value=None, description='Language')
upload = FileUpload(accept='audio/*', multiple=False, description='Upload')
record_button = Button(description='Record and transcribe', button_style='primary')
upload_button = Button(description='Transcribe upload')
out = Output()

def render_result(result):
    print(json.dumps({k: v for k, v in result.items() if k not in ['text', 'gpu_before', 'gpu_after']}, indent=2))
    print('\nTranscript:')
    print(result['text'])
    display(Audio(result['audio_path']))

def on_record(_):
    with out:
        clear_output()
        path = audio.record_audio(record_seconds.value)
        render_result(asr.transcribe_audio(asr_model, path, language.value))

def on_upload(_):
    with out:
        clear_output()
        path = audio.save_uploaded_audio(upload.value)
        render_result(asr.transcribe_audio(asr_model, path, language.value))

record_button.on_click(on_record)
upload_button.on_click(on_upload)
display(VBox([HBox([record_seconds, language]), HBox([record_button, upload, upload_button]), out]))

In [ ]:
# Load and persist a small homogeneous Spanish sample from the HF dataset.
DATASET_SPLIT = 'validation_ABR'
MIN_SECONDS = 3.0
MAX_SECONDS = 8.0
SAMPLE_SIZE = 8
SEED = 42

dataset_sample = get_data.get_dataset_sample(
    split=DATASET_SPLIT,
    min_seconds=MIN_SECONDS,
    max_seconds=MAX_SECONDS,
    sample_size=SAMPLE_SIZE,
    seed=SEED,
    sample_dir=WORK_DIR / 'dataset_samples',
)

sample_df = get_data.sample_to_dataframe(dataset_sample)
display(sample_df)

In [ ]:
# Explore persisted dataset audio/transcriptions before benchmarking.
get_data.explore_sample(dataset_sample)

In [ ]:
# Benchmark a homogeneous persisted sample from BSC-LT/distilled-yodas-spanish.
REPEATS = 5
WARMUP_RUNS = 1
BATCH_SIZES = [1]

stt_rows, stt_summary = asr.run_stt_load_test(
    asr_model,
    dataset_sample,
    repeats=REPEATS,
    warmup_runs=WARMUP_RUNS,
    batch_sizes=BATCH_SIZES,
    language='Spanish',
)
display(stt_rows)
display(pd.DataFrame([stt_summary]))
stt_rows.to_csv(WORK_DIR / 'stt_load_test_rows.csv', index=False)
pd.DataFrame([stt_summary]).to_csv(WORK_DIR / 'stt_load_test_summary.csv', index=False)
print('Cost formula: eur_per_audio_min = eur_per_runtime_min / audio_minutes_processed_per_runtime_minute')